# Look-ahead · Text  `[EVAL]`

**What did each policy *learn* and *unlearn*, in the text itself?** Every other artifact in the tree reads
the transcripts as per-conversation rates (length, `?` per turn, loops, two lexical markers). This family
reads the utterances as content, in the sentence-embedding space the training-side probe already uses
(`pref.py`, `all-MiniLM-L6-v2`), so the eval side and the update side are comparable:

1. **Repertoire** — k-means on the BASE policy's therapist turns = its repertoire of behaviours; cluster
   *occupancy* per arm × iteration (+ an out-of-repertoire `novel` share). Growing clusters = learned,
   shrinking = unlearned. Stability over `k` and seeds reported.
2. **Drift** — centroid displacement from base; cosine between the two K arms' displacements (same thing
   learned?); cos(update direction, realised drift).
3. **Diversity / persona sensitivity** — template similarity at matched turn index, between-persona
   variance share, near-duplicate rate, distinct-n.
4. **Per-conversation metrics, persona-paired K contrast** — distance to base, novel share,
   self-similarity, semantic echo + lexical recall of the preceding patient turn (judge-free reflection
   proxies, cross-checked against the oracle's MITI reflection counts under both graders), and the
   PATIENT side (turn length, questions, disengagement cue, echo of the therapist).
5. **Within-session profile** — therapist features by turn bin (early / mid / late), endpoint vs base.

**Conventions.** Sign `+ ⇒ K=0 higher` (K=0 − K=5), pairing unit `persona_id`, iteration 0 = the base
policy (two independent draws per method, pooled for the repertoire fit). The scripted therapist opener
(utterance 0) is excluded everywhere — it is the prompt, not the policy. Everything here is judge-free
except §4's cross-check, which loads both graders and never averages them. Holm scope is stated per table.
Bootstraps and k-means seed with `BOOT_SEED`. Lexical cues (`RE_EFFUSIVE`, `RE_AFFIRM`, `RE_DISENGAGE`)
are directional sanity markers, not measurements.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 60, "display.max_colwidth", 120)

import eda_analysis
from eda_analysis import exports, plotting, stats, behavior, lookahead, reliability, text
from eda_analysis.constants import BOOT_SEED, set_active_judge, judge_dirname, PRIMARY_JUDGE_TAG

cfg = eda_analysis.EdaConfig(family="lookahead/text", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp figures/_provenance.md (reset just removed the one notebook_setup wrote)

## 0 · Utterances + embeddings

One row per utterance of every eval conversation (4 arms × 11 states × 96), embedded with the same
MiniLM model and sha1 cache as the training-side probe (`eda/.emb_cache/eval/`). Persona attached per arm
by replaying the per-iteration shuffle, so every per-conversation metric below is pairable across arms.

In [ ]:
KA = eda_analysis.cross_k_arms(S)
K_ARMS = [a.label for a in KA if a.label in ("PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5")]
PAL = plotting.arm_palette(K_ARMS)

def support(frame, **kw) -> str:
    note = eda_analysis.support_note(frame, **kw)
    return f" {note}" if note else ""

t0 = time.time()
UTT = text.load_utterances(KA).reset_index(drop=True)
print(f"utterances: {len(UTT):,} rows ({time.time()-t0:.0f}s) | arms {sorted(UTT.arm.unique())}")
print(UTT.groupby(["arm", "role"]).size().unstack())
assert UTT[UTT.utt_idx == 0].text.nunique() == 1, "the scripted opener should be one string"

t0 = time.time()
E = text.embed_utterances(UTT)
print(f"embedded {E.shape} in {time.time()-t0:.0f}s | zero rows: {(np.abs(E).sum(1) == 0).sum()}")
N_CONTENT = int(text.content_mask(UTT).sum())
print(f"therapist content turns (opener + empties excluded): {N_CONTENT:,}")

## 1 · Repertoire — what the base policy could do, and what each arm kept, grew or dropped

k-means (k = 30, `BOOT_SEED`) on the base policy's therapist turns; personas split in half so the
novelty threshold (5th percentile of held-out nearest-centroid cosine) is out-of-sample. Every turn of
every state is assigned to its nearest base cluster; `novel` = below the threshold. Cluster cards name
each cluster by its most distinctive words, lexical tags, MI word-category hits and three exemplars.

In [ ]:
K_CLUSTERS = 30
REP = text.repertoire_fit(UTT, E, k=K_CLUSTERS, seed=BOOT_SEED)
ASSIGNED = text.repertoire_assign(UTT, E, REP)
OCC = text.repertoire_occupancy(ASSIGNED, K_CLUSTERS)
REP_STATE = text.repertoire_by_state(OCC, K_CLUSTERS)
LABELS = text.repertoire_labels(UTT, E, ASSIGNED, REP)
LU = text.learned_unlearned(REP_STATE, LABELS, K_CLUSTERS, top=6)
print(f"fit on {REP['n_fit']:,} base turns, threshold calibrated on {REP['n_calib']:,} held-out base turns "
      f"→ novelty threshold cos = {REP['threshold']:.3f}")
display(REP_STATE[["arm", "iteration", "novel_share", "entropy", "eff_clusters", "n_turns"]].round(3))
display(LABELS[["cluster", "n_base", "share_base", "top_words", "q_rate", "effusive_rate", "affirm_rate", "mean_chars"]].round(3))
display(LU[["arm", "direction", "cluster", "share_base", "share_final", "delta", "top_words"]].round(3))

exports.save_table(LABELS, "repertoire_clusters", caption=(
    f"Cluster cards for the {K_CLUSTERS}-cluster base repertoire (k-means on the base policy's therapist turns, "
    f"{REP['n_fit']:,} fit / {REP['n_calib']:,} calibration turns). top_words = log-odds-distinctive words vs the "
    "rest of the base pool; q_rate / effusive_rate / affirm_rate = share of the cluster's base turns with a '?' / "
    "the RE_EFFUSIVE cue / the RE_AFFIRM cue; mi_* = share containing a word of that MI category list "
    "(pref.MI_CATEGORIES); exemplars = the three base turns nearest the centroid."))
exports.save_table(REP_STATE.round(4), "repertoire_by_state", caption=(
    "Per (arm, iteration): mean cluster share c0..c29 over the state's 96 conversations, novel_share (turns "
    f"below the held-out novelty threshold cos = {REP['threshold']:.3f}), occupancy entropy (nats) and the "
    "effective number of clusters exp(H). Iteration 0 = the base policy, novel_share out-of-sample only for the "
    "calibration half." + support(REP_STATE)))
exports.save_table(LU.round(4), "repertoire_learned_unlearned", caption=(
    "Per arm: the six clusters that grew most (learned) and shrank most (unlearned) in share of therapist turns, "
    "final iteration vs the arm's own base. delta in share-of-turns; exemplar = the base turn nearest the centroid."))

fig = plotting.repertoire_occupancy_fig(REP_STATE, LABELS, K_CLUSTERS, arms=K_ARMS)
exports.save_fig(fig, "repertoire_occupancy", caption=(
    "Base-repertoire occupancy by iteration, one panel per arm: share of therapist turns in each of the ten "
    "largest base clusters (named by distinctive words), the remaining clusters pooled, and the out-of-repertoire "
    "novel share on top. A cluster that fills the panel is a behaviour the policy converged on; a cluster that "
    "vanishes is unlearned." + support(REP_STATE)))
plt.show()
fig = plotting.learned_unlearned_fig(LU, arms=K_ARMS)
exports.save_fig(fig, "repertoire_learned_unlearned", caption=(
    "Δ share of therapist turns (final − base) for each arm's six most-grown (green) and six most-shrunk (orange) "
    "base clusters. Read with repertoire_clusters for the exemplars."))
plt.show()

t0 = time.time()
STAB = text.repertoire_stability(UTT, E, ks=(20, 30, 40), seeds=(0, 1, 2), ref_k=K_CLUSTERS, ref_seed=BOOT_SEED)
print(f"stability sweep in {time.time()-t0:.0f}s")
display(STAB.round(3))
exports.save_table(STAB.round(4), "repertoire_stability", caption=(
    f"Robustness of the repertoire reading to k ∈ {{20, 30, 40}} and the k-means seed: Spearman ρ of the per-state "
    f"novel_share and entropy series against the reference fit (k = {K_CLUSTERS}, seed = BOOT_SEED), and the adjusted "
    "Rand index of the base-pool labels vs the reference at the same k. Per-state scalars are the quantities "
    "comparable across k; cluster identities are not."))

## 2 · Drift — where each policy moved, and whether the two K arms moved alike

Centroid of the therapist content turns per state; displacement = centroid − pooled base centroid.
`cos_K0_K5` says whether the two arms of a method learned the *same* thing; `cos_PTO_GRPO` the same across
optimisers at one K. The alignment table puts the training-side update direction (`pref.direction_by_arm`,
same embedding space) next to the realised drift: does the policy move where the update pushed?

In [ ]:
CENTS = text.state_centroids(UTT, E)
DRIFT, POOLED_BASE = text.drift_by_state(CENTS)
COS = text.drift_cosines(CENTS, POOLED_BASE)
display(DRIFT.round(4)); display(COS.round(4))
exports.save_table(DRIFT.round(4), "drift_by_state", caption=(
    "Per (arm, iteration): drift_norm = ‖centroid − pooled base centroid‖ of the therapist content turns (unit "
    "MiniLM vectors, so the scale is cosine-like), drift_norm_own = vs the arm's own iteration-0 centroid, step_norm = "
    "‖c_it − c_it−1‖, cos_to_final = cos(d_it, d_final) (1 = the path is a straight line)." + support(DRIFT)))
exports.save_table(COS.round(4), "drift_cosines", caption=(
    "Per iteration: cosine between displacement vectors (centroid − pooled base) — the two K arms of one method "
    "(cos_K0_K5_*) and the two methods at one K (cos_PTO_GRPO_K*). 1 = moved the same way, 0 = orthogonal."))

t0 = time.time()
try:
    ALIGN = text.update_alignment(CENTS, POOLED_BASE, KA)
    print(f"update-direction alignment in {time.time()-t0:.0f}s")
except Exception as ex:     # the training logs may be unreachable on a machine without the Drive mount
    print(f"WARNING: update_alignment unavailable ({type(ex).__name__}: {ex}) — table written empty")
    ALIGN = pd.DataFrame()
display(ALIGN.round(4))
exports.save_table(ALIGN.round(4), "update_alignment", caption=(
    "cos(update direction, realised drift) per (arm, iteration ≥ 1): the arm's pooled training-side update "
    "direction (pref.direction_by_arm, MiniLM space, Σ|w| = 2 weights) vs the cumulative eval-side drift c_it − "
    "c_base (cos_pooled_vs_cumdrift); that iteration's own update direction vs the step c_it − c_it−1 "
    "(cos_iter_vs_step) and vs the cumulative drift. Per-iteration PTO directions are noisy (split-half ≈ 0.2; "
    "arms/preference/tables/*/update_direction_quality.md) — read those rows with that in mind."))
fig = plotting.drift_fig(DRIFT, COS, ALIGN if not ALIGN.empty else None, arms=K_ARMS, palette=PAL)
exports.save_fig(fig, "drift", caption=(
    "(a) how far each policy's therapist-turn centroid moved from the pooled base centroid, by iteration; (b) the "
    "cosine between displacement vectors — the two K arms of one method, and the two methods at one K; (c) the "
    "cosine between each arm's pooled training-side update direction and its realised drift." + support(DRIFT)))
plt.show()

## 3 · Diversity and persona sensitivity

Does training collapse the therapist onto a template? `template_sim` = mean pairwise cosine across
personas at the same turn index; `persona_var_share` = the between-conversation share of embedding variance
(bootstrap CI over conversations) — a therapist that tailors to the patient keeps it high; `dup_rate` = share
of turns with a ≥ 0.95-cosine twin in another conversation of the same state; `distinct_n` on a fixed
400-turn sample so length cannot drive it.

In [ ]:
t0 = time.time()
DIV = text.diversity_by_state(UTT, E, n_boot=300, seed=BOOT_SEED)
TST = text.template_similarity_by_turn(UTT, E)
print(f"diversity in {time.time()-t0:.0f}s")
display(DIV[["arm", "iteration", "template_sim", "persona_var_share", "persona_var_share_lo", "persona_var_share_hi",
             "dup_rate", "distinct_1", "distinct_2", "distinct_3", "mean_words", "n_turns"]].round(4))
exports.save_table(DIV.round(4), "diversity_by_state", caption=(
    "Per (arm, iteration), over the state's therapist content turns: template_sim (mean pairwise cosine across "
    "conversations at the same therapist turn index 1..8, averaged over indices), persona_var_share (between-"
    "conversation / total embedding variance, 95% bootstrap CI over conversations, BOOT_SEED), dup_rate (share of "
    "turns whose nearest neighbour in ANOTHER conversation of the state has cosine ≥ 0.95), distinct_1/2/3 (unique / "
    "total n-grams over a fixed random sample of 400 turns), mean_words." + support(DIV)))
exports.save_table(TST.round(4), "template_similarity_by_turn", caption=(
    "Long form of template_sim: per (arm, iteration, therapist turn index 1..10) the mean pairwise cosine across "
    "conversations at that index, with n = conversations reaching it."))
fig = plotting.diversity_fig(DIV, arms=K_ARMS, palette=PAL)
exports.save_fig(fig, "diversity", caption=(
    "Diversity and persona sensitivity by iteration: template similarity, between-persona variance share (band = "
    "95% bootstrap CI), near-duplicate rate, distinct-2. Rising template similarity with falling variance share and "
    "distinct-2 is a policy converging on one script regardless of the patient." + support(DIV)))
plt.show()

## 4 · Per-conversation metrics — levels, persona-paired K contrast, and the reflection cross-check

`echo` = cos(therapist turn, preceding patient turn) − mean cos to 32 random patient turns of the same
state; `lex_recall_prev` = share of the preceding patient turn's content words re-used by the therapist.
Both are judge-free candidates for a *reflection* proxy, so §4b cross-checks them against the oracle's MITI
reflection counts under BOTH graders (and against question counts as a discriminant check), the way the
`?`-rate is cross-checked against `MITI_B3_Q`. The patient side: turn length, `?` per patient turn, the
disengagement cue, and the patient's echo of the therapist.

In [ ]:
t0 = time.time()
FEATS = text.utterance_features(UTT, E, seed=BOOT_SEED)
PC = text.per_conv_metrics(UTT, E, POOLED_BASE, feats=FEATS)
PC = PC.merge(OCC[text._CONV + ["novel_share"]], on=text._CONV, how="left")
print(f"per-conversation metrics {PC.shape} in {time.time()-t0:.0f}s")
LEVELS = text.state_table(PC, text.TEXT_K_METRICS)
display(LEVELS[["arm", "iteration"] + text.TEXT_K_METRICS].round(4))
exports.save_table(LEVELS.round(4), "text_levels", caption=(
    "Per (arm, iteration): mean and SE over 96 conversations of the per-conversation text metrics — "
    + "; ".join(f"{m} = {d}" for m, d in text.TEXT_METRIC_LABELS.items()) + "." + support(LEVELS)))

SL = text.to_scores_long(PC, KA, text.TEXT_K_METRICS)
KT = lookahead.paired_k_frames({lookahead.TEXT_JUDGE_LABEL: SL}, metrics=text.TEXT_K_METRICS, holm_family="iterations")
KT["lower_better"] = KT["metric"].isin(text.LOWER_BETTER)
KTS = lookahead.k_summary(KT) if not KT.empty else pd.DataFrame()
display(KT[["method", "metric", "iteration", "n", "mean_K0", "mean_K5", "mean_delta", "dz", "p", "p_holm", "sig"]].round(4))
display(KTS)
exports.save_table(KT.round(4), "k_text_paired", caption=(
    "Persona-paired K=0 − K=5 on every per-conversation text metric, per method and matched iteration (n = 96 "
    f"persona pairs). {lookahead.SIGN_NOTE} Holm across ITERATIONS within (method, metric). lower_better marks "
    "pt_disengage_rate and within_sim." + support(SL, subject="no later matched iteration")))
exports.save_table(KTS, "k_text_summary", caption=(
    "Per (method, metric): tally of matched iterations where K=0 − K=5 is significant after Holm, by sign, with the "
    "mean dz — the summary of k_text_paired."))
fig = plotting.k_text_forest(KT, lower_better=text.LOWER_BETTER, labels={m: m for m in text.TEXT_K_METRICS})
exports.save_fig(fig, "k_text_forest", caption=(
    "Persona-paired dz of K=0 − K=5 on every text metric at each method's last matched iteration; lower-better "
    "metrics sign-flipped so a bar to the right always reads 'K=0 better'. Holm stars across iterations within "
    "(method, metric)."))
plt.show()
fig = plotting.levels_fig(LEVELS, text.TEXT_K_METRICS, text.TEXT_METRIC_LABELS, arms=K_ARMS, palette=PAL,
                          lower_better=text.LOWER_BETTER)
exports.save_fig(fig, "text_levels", caption=(
    "The per-conversation text metrics by iteration (mean ± SE over 96 personas), one panel each, all four arms."
    + support(LEVELS)))
plt.show()

In [ ]:
# 4b · Cross-check the judge-free reflection proxies against the oracle's MITI reflection counts, per grader.
TAGS = {judge_dirname(t): t for t in reliability.second_judge_tags()}
TAGS[judge_dirname(PRIMARY_JUDGE_TAG)] = ""
VAL, POOLED = [], []
try:
    for j, tag in TAGS.items():
        set_active_judge(tag, 0)
        miti = behavior.load_miti_behavior(KA, attach_persona=False)
        if miti.empty:
            print(f"{j}: no MITI scores — skipped"); continue
        v = text.echo_validation(PC, miti).assign(judge=j)
        VAL.append(v); POOLED.append(text.pooled_rho(v).assign(judge=j))
finally:
    set_active_judge("", 0)
VAL = pd.concat(VAL, ignore_index=True) if VAL else pd.DataFrame()
POOLED = pd.concat(POOLED, ignore_index=True) if POOLED else pd.DataFrame()
display(POOLED.round(3))
exports.save_table(VAL.round(4), "echo_validation", caption=(
    "Per (grader, proxy, arm, iteration): Spearman ρ across the 96 conversations between a judge-free responsiveness "
    "proxy (echo = baseline-subtracted cosine to the preceding patient turn; lex_recall_prev = share of its content "
    "words re-used) and the grader's MITI reflections per therapist turn ((B4_SR + B5_CR) / n_th_turns), complex "
    "reflections alone, and questions per turn as a discriminant. A reflection proxy should track reflections and "
    "not questions."))
exports.save_table(POOLED.round(4), "echo_validation_pooled", caption=(
    "Fisher-z pooled within-state ρ per (grader, proxy, method) — the number to quote for whether either proxy "
    "is a usable judge-free reflection channel. Near zero = it is a responsiveness measure, not a reflection measure."))

## 5 · Within-session profile

The same therapist features by turn bin (1–2 / 3–5 / 6–9 / 10+ of the therapist's own turns) at each
arm's endpoint against the pooled base. Where in the session does over-praise concentrate? Do the K arms
differ early or only once the session runs long? The bin contrast is persona-paired at the last matched
iteration, Holm across bins within (method, feature).

In [ ]:
PROF = text.session_profile(UTT, FEATS)
PROF_CONV = text.profile_per_conv(UTT, FEATS)
PKC = text.profile_kcontrast(PROF_CONV, KA, features=("n_chars", "q_count", "effusive", "echo", "lex_recall_prev"))
FINAL = {a: int(PROF[PROF.arm == a].iteration.max()) for a in K_ARMS}
display(PROF[PROF.iteration.isin([0] + list(FINAL.values()))][["arm", "iteration", "bin"] + text.PROFILE_FEATURES + ["n_turns", "share_convs_reaching"]].round(3))
display(PKC.round(4))
exports.save_table(PROF.round(4), "session_profile", caption=(
    "Per (arm, iteration, therapist turn bin): mean n_chars, '?' count, RE_EFFUSIVE and RE_AFFIRM cue rates, echo "
    "and lexical recall of the preceding patient turn, n_turns, and the share of the state's conversations that "
    "reach the bin. Opener excluded." + support(PROF)))
exports.save_table(PKC.round(4), "profile_kcontrast", caption=(
    "Persona-paired K=0 − K=5 per (method, feature, turn bin) at the last matched iteration, on per-conversation "
    f"bin means. {lookahead.SIGN_NOTE} Holm across bins within (method, feature)."))
fig = plotting.profile_fig(PROF, text.PROFILE_FEATURES, arms=K_ARMS, iteration_by_arm=FINAL, palette=PAL)
exports.save_fig(fig, "session_profile", caption=(
    "Within-session profile at each arm's endpoint (coloured; K=0 solid, K=5 dashed) against the pooled base "
    "(grey dotted): therapist turn length, '?' count, effusive-cue rate, affirmation-cue rate, semantic echo and "
    "lexical recall of the preceding patient turn, by therapist turn bin."))
plt.show()

In [ ]:
NUM = text.text_numbers(REP_STATE, DRIFT, COS, DIV, LEVELS, REP)
if not POOLED.empty:
    for _, r in POOLED.iterrows():
        NUM[f"echo_validation.{r['judge']}.{r['proxy']}.{r['method']}.rho_vs_reflections"] = {
            "value": round(float(r["rho_vs_reflections"]), 4), "source": "echo_validation_pooled",
            "note": "Fisher-z pooled within-state Spearman ρ"}
exports.save_numbers("text_numbers", NUM, caption=(
    "Ledger of the family's endpoint scalars (repertoire novel share / entropy, drift norms and cosines, diversity, "
    "text-metric levels, pooled echo-validation ρ), each citing its source table."))
print(f"{len(NUM)} ledger keys")

In [ ]:
exports.prune_orphan_captions(); exports.build_index()